# LifeMate — GitHub → planner training → submission
Select a **GPU runtime**, then run the cells in order. This notebook clones your repository and calls Python modules in `training_pipeline/`; it contains configuration and stage commands only.

**Pipeline:** teammate ASR → BGE history/tool ranking → Qwen3-4B-Instruct-2507 tool execution → submission. Only Qwen's LoRA adapters are trained. ASR and BGE remain frozen.

**Before Colab:** commit and push `training_pipeline/` together with this notebook to the repository/ref configured below. No embedded source generation is needed.

Drive outputs: `MyDrive/Zalo-AI-Challenge-2026/runs/<RUN_NAME>/best_model/` and `MyDrive/Zalo-AI-Challenge-2026/submission.jsonl`. Your original `lifemate_colab.ipynb` remains unchanged.

The source recipe reported 53.40 validation / 59.81 held-out test with gold transcripts; these are not public leaderboard results or a promised score. See `notebooks/train_and_infer_review.md` for the reference review and local verification.


## 1. Mount Drive and set paths
Keep the same run name to resume the **same** configuration; use a new name for a different recipe. Checkpoints and reports are written directly to Drive. Dataset, environments and SQLite cache stay on local Colab disk.

ASR can be supplied as a JSONL from **any architecture** (including Gipformer), or generated in a later cell from a merged Whisper-compatible checkpoint. Training needs neither. Set `ASR_MODE` and the relevant path before the public stages.

**Training across days:**
- First day: `TRAIN_START='fresh'`, `RUN_NAME='day1'`, `EPOCHS=2`.
- Next day: `TRAIN_START='best'`, `PREVIOUS_RUN_NAME='day1'`, `RUN_NAME='day2'`, `EPOCHS=2`.
- Following day: use `PREVIOUS_RUN_NAME='day2'` with another new run name.

`best` loads the previous selected LoRA weights and trains for **additional** epochs with a fresh optimizer and learning-rate schedule. It is a new fine-tuning session, not an exact continuation of a single 10-epoch schedule. The previous best also competes in validation selection, so a worse new epoch does not replace it. Keep the validation split fixed; reserve held-out test evaluation for the final session.

For an interrupted session, keep its `RUN_NAME`, start settings and epoch count unchanged; `RESUME_CHECKPOINT='auto'` restores the full latest checkpoint, including optimizer/scheduler. A completed run is not extended in place: start a new run from its best instead. For a new fresh start, use a new run name.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, subprocess
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/Zalo-AI-Challenge-2026')
RUN_NAME = 'qwen3_4b_compact_sft_v1'
EPOCHS = 2  # Additional epochs for this run when TRAIN_START='best'.
TRAIN_START = 'fresh'  # 'fresh': base model; 'best': previous run's selected adapter.
PREVIOUS_RUN_NAME = 'qwen3_4b_compact_sft_v1'  # For 'best'; use a NEW RUN_NAME above.
RUN_BASELINE = False  # Optional untrained planner score on the SAME validation split.
ASR_MODE = 'transcripts'  # 'transcripts' or 'whisper'
ASR_TRANSCRIPTS = DRIVE_ROOT / 'inputs/public_asr.jsonl'
ASR_MODEL = ''  # Only for whisper: teammate's merged checkpoint folder / Hub ID.
EXPORT_BASE_WEIGHTS = True  # Self-contained best_model bundle; needs several GB on Drive.
RESUME_CHECKPOINT = 'auto'  # Resume an interruption within this SAME run, including optimizer/scheduler.

WORK_ROOT = Path('/content/lifemate_train_infer')
OUTPUT_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
for folder in (DRIVE_ROOT / 'inputs', WORK_ROOT, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = OUTPUT_DIR / 'run_config.json'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Persistent run:', OUTPUT_DIR)


### Clone the implementation
Push the new files before the first Colab run. To receive later updates, explicitly refresh the checkout in a new runtime; this cell does not automatically pull changes into an active training session.


In [ ]:
# Clone once. Existing checkouts are retained so reruns do not silently change code.
REPO_URL = 'https://github.com/votaquangnhat/LifeMate-Baseline.git'
REPO_REF = 'main'  # Use a commit SHA for reproducible runs.
PROJECT = Path('/content/lifemate-training')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', REPO_REF], check=True)
CODE_DIR = PROJECT / 'training_pipeline'
assert (CODE_DIR / 'colab_workflow.py').is_file(), 'Push training_pipeline/ to GitHub first, then use a fresh clone containing those files.'
revision = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
expected = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', REPO_REF], text=True).strip()
assert revision == expected, 'Existing checkout differs from REPO_REF; explicitly select the intended revision before continuing.'
revision_file = OUTPUT_DIR / 'repository_revision.txt'
if revision_file.exists():
    assert revision_file.read_text().strip() == revision, 'Repository revision changed: use a new RUN_NAME.'
revision_file.write_text(revision + '\n')
sys.path.insert(0, str(CODE_DIR))
import colab_workflow as workflow
print('Repository:', PROJECT, 'commit:', revision)


## 2. Download and extract the same dataset
The Drive file ID below comes from your baseline. Alternatively put the ZIP at `LOCAL_ARCHIVE` before running. Data is extracted locally; it is not included in the model export.


In [ ]:
DATA_ROOT = workflow.prepare_data(WORK_ROOT)


## 3. Install an isolated agent runtime
Uses Colab’s CUDA PyTorch, with a separate Python environment for agent packages. Each GPU stage runs in a fresh subprocess so the reranker, trainer and ASR do not occupy VRAM together. Model revisions and package versions are saved; resuming requires matching versions. No token or credential is written to the notebook.

The source recipe uses Unsloth; its [Qwen training interface](https://github.com/QwenLM/Qwen3/blob/main/docs/source/training/unsloth.md) and the [Transformers Trainer API](https://huggingface.co/docs/transformers/main_classes/trainer) are the relevant upstream interfaces. The GPU preflight is required because local CPU checks cannot certify your Colab runtime.


In [ ]:
AGENT_PYTHON = workflow.setup_environment(WORK_ROOT, OUTPUT_DIR)


## 4. Training configuration
The BGE reranker retains short schemas for **all** installed tools, prioritizing relevant descriptions; it keeps recent history and retrieves relevant older blocks. No gold action is used to choose input schemas/history. Prompts, targets and current tool observations are not silently truncated. Preparation measures actual lengths; preflight checks the resulting memory demand.

The split is grouped by shared conversations/audio (85/10/5), matching the completed teammate recipe. Share `split_assignments.json` with your ASR teammate if you want an uncontaminated end-to-end held-out comparison. Gold-transcript scores measure only the planner.


In [ ]:
CONFIG = workflow.configure(
    DATA_ROOT=DATA_ROOT, OUTPUT_DIR=OUTPUT_DIR, CODE_DIR=CODE_DIR, CONFIG_PATH=CONFIG_PATH,
    WORK_ROOT=WORK_ROOT, DRIVE_ROOT=DRIVE_ROOT, RUN_NAME=RUN_NAME,
    TRAIN_START=TRAIN_START, PREVIOUS_RUN_NAME=PREVIOUS_RUN_NAME,
    EPOCHS=EPOCHS, RESUME_CHECKPOINT=RESUME_CHECKPOINT, ASR_TRANSCRIPTS=ASR_TRANSCRIPTS,
)


## 5. Check the repository runtime
Run the CPU contract tests before loading model weights. A source snapshot is saved to the run folder for provenance; execution uses the cloned `training_pipeline/` files. The notebook does not write Python implementations.


In [ ]:
workflow.check_runtime(AGENT_PYTHON, CODE_DIR, OUTPUT_DIR)

def run_stage(stage):
    workflow.run_stage(AGENT_PYTHON, CODE_DIR, CONFIG_PATH, stage)


## 6. Prepare SFT and evaluate schema coverage
This replays labeled **training** trajectories to obtain real observations and reference hashes. It produces assistant-only targets for each next action and `done`; user text, schemas, history and tool results are masked from loss. All train/dev/test data stays separate. Nothing from public labels or environment files is read by the agent.

Inspect `retrieval_report.json` and `length_report.json`. Schema coverage must be 100%; top-k description recall is a separate diagnostic. Cached preparation is reused only if code, inputs, tokenizer, packages and configuration match.


In [ ]:
run_stage('prepare')
CONFIG = json.loads(CONFIG_PATH.read_text())
for name in ('data_summary.json', 'length_report.json', 'retrieval_report.json'):
    print(name, (OUTPUT_DIR / name).read_text())


## 7. GPU preflight
Five optimizer updates exercise representative and longest sequences with real accumulation. FP16 is used on T4; BF16 is used when supported on L4/A100. The preflight checks loss, actual successful updates and peak VRAM, then discards those weights. If it fails, inspect the log before training; no silent target truncation or batch change occurs.


In [ ]:
run_stage('smoke')
print((OUTPUT_DIR / 'preflight_report.json').read_text())


## 8. Optional baseline, then train
Training saves full resume checkpoints every 100 optimizer steps and at each epoch boundary directly to Drive. `auto` resumes the latest complete checkpoint after a disconnect. If the notebook is interrupted, reconnect, rerun the setup/preparation cells, and continue. A GPU/packaging mismatch is reported rather than silently changing the run.

Selection evaluates epoch checkpoints only; periodic checkpoints are for resuming. Disk usage can grow because optimizer states are retained.

When `TRAIN_START="best"`, both preflight and actual training start from the snapshotted previous best adapter. Its weight hash is part of the training signature; full interrupted-session checkpoints take precedence when resuming.


In [ ]:
if RUN_BASELINE:
    run_stage('baseline')
    print((OUTPUT_DIR / 'baseline_metrics.json').read_text())
run_stage('train')
print((OUTPUT_DIR / 'training_complete.json').read_text())


## 9. Select the best epoch by full validation rollout
Every validation sample runs through ToolEnv, including the model’s own intermediate decisions. Selection uses the teammate’s reconstruction of the competition metric (state-hash gate and one-to-one tool-call matching), not teacher-forced loss. The source field `official_score_100` is a local reconstruction, not an official leaderboard result. Ties choose the earliest checkpoint. Internal test scores do not select the model.

In `best` mode, the incoming previous best is evaluated on the same validation split alongside the new epoch checkpoints. Ties retain the previous best.


In [ ]:
run_stage('select')
print((OUTPUT_DIR / 'selection.json').read_text())


## 10. Save the best model to Drive now
`best_model/adapter/` holds the selected trained weights and tokenizer. By default this cell also saves the exact NF4 base and BGE weights, so the bundle contains the required model weights as well as runtime source/configuration. This is a **base + LoRA bundle**, not merged full-precision weights. The model and adapter must be loaded together using the supplied runtime. If `EXPORT_BASE_WEIGHTS=False`, the pinned Hub models remain required at inference.

The saved `deployment_config.json` points to the exported bundle on this Drive. For another machine, update paths. ASR is supplied separately. Model export does not include competition data or ASR weights.


In [ ]:
BEST_MODEL = workflow.export_model(
    AGENT_PYTHON, OUTPUT_DIR, CODE_DIR, CONFIG_PATH,
    EXPORT_BASE_WEIGHTS=EXPORT_BASE_WEIGHTS,
)


## 11. Held-out planner evaluation
Run after model selection. This uses gold transcripts and does not measure ASR quality. `planner_seconds` excludes ASR and reranking; it does not certify the end-to-end 30-second requirement or final Docker deployment. Review aborted rollouts and tool errors in the saved traces.

For daily continued training, skip this cell until your final session; use validation alone for intermediate decisions.


In [ ]:
run_stage('test')
print((OUTPUT_DIR / 'test_metrics.json').read_text())


## 12. ASR handoff — no training
Your teammate can produce `inputs/public_asr.jsonl` with one record per public ID:

```json
{"id":"public_0001","audio_text":"recognized final request","history_audio_texts":["first user history turn","second user history turn"]}
```

`history_audio_texts` contains **only user turns**, in their original order; use `[]` for no user history. A dictionary keyed by exact relative audio paths is also accepted. Blank/missing/duplicate records fail before any public environment is created. Their ASR may do its own Vietnamese ITN; preserve names and quoted message text. This notebook does not blindly rewrite numbers inside free-text messages.

The first cell exports a manifest of audio paths for the teammate. The second either validates their file or runs inference from a merged Whisper-compatible checkpoint. For Gipformer or another architecture, use transcript mode; do not pass its checkpoint to the Whisper loader.


In [ ]:
workflow.write_asr_manifest(DATA_ROOT, DRIVE_ROOT, ASR_TRANSCRIPTS)


In [ ]:
workflow.prepare_asr(
    AGENT_PYTHON, CODE_DIR, DATA_ROOT, CONFIG_PATH,
    ASR_MODE, ASR_TRANSCRIPTS, ASR_MODEL,
)


## 13. Run public inference and validate the submission
Loads the selected adapter, compiles ASR/history with the same preprocessing used for SFT, executes one action at a time, and writes calls and hashes from the actual ToolEnv. All executed calls remain in the submission, including rejected calls.

A persistent journal skips completed samples on rerun. If interruption occurs **inside** a sample, it refuses to reset that public environment and reports the unresolved ID; automatic replay would violate the dataset’s single-session rule. Inspect/retain the journal rather than deleting it to force a retry. Schema validation checks format/coverage, not prediction accuracy.


In [ ]:
run_stage('public')
SUBMISSION = workflow.publish_submission(
    OUTPUT_DIR, DRIVE_ROOT, DATA_ROOT, RUN_NAME, BEST_MODEL, ASR_TRANSCRIPTS,
)


In [ ]:
# Optional browser download; the Drive copies already persist.
from google.colab import files
files.download(str(DRIVE_ROOT / 'submission.jsonl'))


## Implementation and verification
Training/runtime code and CPU tests live in `training_pipeline/`. Review details and known limits are in `notebooks/train_and_infer_review.md`.

Use `TRAIN_START='fresh'` with a new run name to start from the base model. Use `TRAIN_START='best'`, yesterday's `PREVIOUS_RUN_NAME`, and a new run name to train additional epochs. Full interrupted-session resume remains separate via `RESUME_CHECKPOINT='auto'`.

GPU training and public scoring still require the Colab run. The code refactor does not change the model recipe, preprocessing, or scoring.
